# Block 1 Solution — Read, Explore, and Report

Completed version of `01_read_explore/read_explore_exercise.ipynb`. The markdown notes at each
TODO explain the approach.

In [ ]:
import odmlib.define_loader as DL
import odmlib.loader as LD

loader = LD.ODMLoader(DL.XMLDefineLoader(model_package="define_2_1"))
loader.open_odm_document("../data/defineV21-SDTM.xml")

odm = loader.root()                  # call root() once and keep the reference
mdv = odm.Study.MetaDataVersion      # single objects in Define-XML - no [0]

print("Study:      ", odm.Study.GlobalVariables.StudyName)
print("MetaDataVer:", mdv.Name)
print("Define ver: ", mdv.DefineVersion)

In [ ]:
for std in mdv.Standards.Standard:
    print(f"{std.OID:<10} {std.Name:<10} {std.Version:<12} ({std.Type})")

In [ ]:
for igd in mdv.ItemGroupDef:
    print(f"{igd.OID:<12} {igd.Name:<8} {len(igd.ItemRef):>3} variables   {igd.Structure}")

## TODO 1 — Find a variable and describe it

`find(class_name, attribute, value)` searches all descendants and returns the first match (or
`None`). One call replaces a hand-written nested loop, and it works from any element — here we
search from the `MetaDataVersion`.

In [ ]:
age = mdv.find("ItemDef", "OID", "IT.DM.AGE")

print("Name:       ", age.Name)
print("DataType:   ", age.DataType)
print("Length:     ", age.Length)
print("Description:", age.Description.TranslatedText[0]._content)
print("Origin:     ", age.Origin[0].Type)

## TODO 2 — List a dataset's variables

An `ItemRef` holds the dataset-specific facts (order, mandatory, key sequence) and points at
an `ItemDef` by OID for the variable-level facts — so a listing is a sort plus one `find()`
per reference.

In [ ]:
vs = mdv.find("ItemGroupDef", "OID", "IG.VS")

print(f"{'#':>3} {'Name':<10} {'Type':<10} {'Len':>4} {'Mandatory'}")
for ref in sorted(vs.ItemRef, key=lambda r: int(r.OrderNumber)):
    item = mdv.find("ItemDef", "OID", ref.ItemOID)
    length = item.Length if item.Length is not None else ""
    print(f"{ref.OrderNumber:>3} {item.Name:<10} {item.DataType:<10} {length!s:>4} {ref.Mandatory}")

## TODO 3 — Decode a codelist

Two reference hops: `ItemDef → CodeListRef.CodeListOID → CodeList`. This codelist uses
`CodeListItem` (coded value + decode); enumerated codelists would use `EnumeratedItem`
(coded value only), so robust code checks both.

In [ ]:
vstestcd = mdv.find("ItemDef", "OID", "IT.VS.VSTESTCD")
cl = mdv.find("CodeList", "OID", vstestcd.CodeListRef.CodeListOID)

print(f"{vstestcd.Name} uses codelist {cl.OID} ({cl.Name}):")
for term in cl.CodeListItem:
    print(f"   {term.CodedValue:<10} = {term.Decode.TranslatedText[0]._content}")
for term in cl.EnumeratedItem:
    print(f"   {term.CodedValue}")

## TODO 4 — Basic define.xml metrics

Every metadata collection on `MetaDataVersion` is a plain Python list, so a document profile
is just `len()` calls — no XPath, no namespace handling.

In [ ]:
metrics = {
    "datasets": len(mdv.ItemGroupDef),
    "variables": len(mdv.ItemDef),
    "codelists": len(mdv.CodeList),
    "methods": len(mdv.MethodDef),
    "value lists": len(mdv.ValueListDef),
    "where clauses": len(mdv.WhereClauseDef),
    "comments": len(mdv.CommentDef),
    "documents": len(mdv.leaf),
}

for name, count in metrics.items():
    print(f"{name:>14}: {count}")

Expected: 11 datasets, 179 variables, 40 codelists, 33 methods, 8 value lists, 32 where
clauses, 29 comments, 3 documents.

## Stretch — the same pattern, a different standard

Only the loader class changes; navigation and `find()` work exactly the same on the ARM model.

In [ ]:
import odmlib.arm_loader as AL

arm_loader = LD.ODMLoader(AL.XMLArmLoader())
arm_loader.open_odm_document("../data/definev21-adam.xml")
arm_mdv = arm_loader.root().Study.MetaDataVersion

for rd in arm_mdv.AnalysisResultDisplays.ResultDisplay:
    print(rd.OID)
    print("   ", str(rd.Description.TranslatedText[0]))
    for ar in rd.AnalysisResult:
        print("     result:", ar.OID)